In [ ]:
# I24BASEPROBE — pulls the code from GitHub so a run is reproducible from a commit SHA
import os, subprocess, sys, shutil, time
REPO = "https://github.com/Ahmadrezanourozii/Automatic-detection-of-diabetic-retinopathy-and-grading-of-diabetic-macular-edema-using-CNN.git"
COMMIT = "568ee7e4135ece035ccb7fad0fd623f333bfb1a9"
WORK = "/kaggle/working/repo"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO, WORK], check=True)
if COMMIT and COMMIT != "HEAD":
    subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
sha = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"]).decode().strip()
print("CODE COMMIT", sha)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--pretty=%s"]).decode().strip())


In [ ]:
import os
for d in sorted(os.listdir("/kaggle/input")):
    n = sum(len(f) for _, _, f in os.walk(f"/kaggle/input/{d}"))
    print(f"{d:55s} {n:7d} files")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
# The pool does not always honour the pinned accelerator. A P100 is sm_60 and the
# preinstalled torch cu128 build ships sm_70+ kernels only, so every CUDA call fails.
# Rather than lose the run, install a torch that supports this device (ISSUES.md §9).
import subprocess, sys, torch
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU {name}  sm_{cap[0]}{cap[1]}  torch {torch.__version__}")
ok = True
try:
    (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
    print("kernels execute fine on this device")
except Exception as e:
    ok = False
    print("UNUSABLE:", e)
if not ok:
    print("installing a torch build that supports this GPU ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)
    print("installed -- src/train.py runs in a subprocess so it picks up the new build")


In [ ]:
import subprocess, sys, os, time, shutil
RUN_ID = "I24BASEPROBE"
OUT = f"/kaggle/working/{RUN_ID}"
# Kaggle carries /kaggle/working across notebook versions (ISSUES.md §13), and this
# run-id already holds the output of a previous version that ran the WRONG script
# (ISSUES.md §24): a results.json, best_*.pt and oof_*.npz from src/train.py. If a later
# version of this notebook fails, Kaggle serves the last COMPLETED version's output, and
# fetch.py's commit check cannot tell the two apart because both pin the same SHA. So the
# directory is emptied before this script runs: whatever ends up here came from this run.
if os.path.isdir(OUT):
    for _stale in sorted(os.listdir(OUT)):
        _p = os.path.join(OUT, _stale)
        print("purging stale artefact from a previous version:", _stale)
        shutil.rmtree(_p) if os.path.isdir(_p) else os.remove(_p)
os.makedirs(OUT, exist_ok=True)
assert not os.listdir(OUT), f"{OUT} is not empty after the purge"
cmd = [sys.executable, "-u", "/kaggle/working/repo/src/retfound_probe.py",
       "--datasets", "/kaggle/input",
       "--splits", "/kaggle/working/repo/data/splits/dev_v1.json",
       "--out", OUT, "--imagenet-baseline", "--batch", "32", "--epochs", "300"]
print(" ".join(cmd), flush=True)
with open(f"{OUT}/run.log", "w") as f:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd="/kaggle/working/repo")
    for line in p.stdout:
        print(line, end="", flush=True); f.write(line); f.flush()
    p.wait()
print("exit=", p.returncode)
